In [1]:
import numpy as np

from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

cancer = load_breast_cancer()

X_data = cancer.data
y_label = cancer.target

X_train, X_test , y_train, y_test = train_test_split(X_data,
                                                     y_label,
                                                     test_size = 0.2,
                                                     random_state = 42)

In [2]:
knn_clf = KNeighborsClassifier(n_neighbors = 4)
rf_clf = RandomForestClassifier(random_state = 42)
dt_clf = DecisionTreeClassifier()
ada_clf = AdaBoostClassifier(n_estimators = 100)

lr_final = LogisticRegression(C = 10)

In [4]:
knn_clf.fit(X_train, y_train);
rf_clf.fit(X_train, y_train);
dt_clf.fit(X_train, y_train);
ada_clf.fit(X_train, y_train);

In [5]:
knn_pred = knn_clf.predict(X_test)
rf_pred = rf_clf.predict(X_test)
dt_pred = dt_clf.predict(X_test)
ada_pred = ada_clf.predict(X_test)

print(accuracy_score(y_test, knn_pred))
print(accuracy_score(y_test, rf_pred))
print(accuracy_score(y_test, dt_pred))
print(accuracy_score(y_test, ada_pred))

0.9385964912280702
0.9649122807017544
0.9298245614035088
0.9736842105263158


In [7]:
# predict 한걸 스태킹 해서 final에 in

pred = np.array([knn_pred, rf_pred, dt_pred, ada_pred])
print(pred.shape)

(4, 114)


In [8]:
pred = np.transpose(pred)
print(pred.shape)

(114, 4)


In [9]:
lr_final.fit(pred, y_test)
final = lr_final.predict(pred)

print(accuracy_score(y_test, final))

0.9736842105263158


In [13]:
from sklearn.model_selection import KFold
import numpy as np

def get_stacking_base_datasets(model, X_train_n, y_train_n, X_test_n, y_test_n, n_folds):
    kf = KFold(n_splits=n_folds, shuffle=False)
    train_fold_pred = np.zeros((X_train_n.shape[0], 1))
    test_pred = np.zeros((X_test_n.shape[0], n_folds))
    print(model.__class__.__name__)

    for folder_counter, (train_index, valid_index) in enumerate(kf.split(X_train_n)):
        print(folder_counter)

        X_tr = X_train_n[train_index]
        y_tr = y_train_n[train_index]
        X_te = X_train_n[valid_index] 

        model.fit(X_tr, y_tr)
        train_fold_pred[valid_index, :] = model.predict(X_te).reshape(-1, 1)
        test_pred[:, folder_counter] = model.predict(X_test_n)

    test_pred_mean = np.mean(test_pred, axis=1).reshape(-1, 1)
    return train_fold_pred, test_pred_mean

In [15]:
knn_train, knn_test = get_stacking_base_datasets(knn_clf, X_train, y_train, X_test, y_test, 7)
rf_train, rf_test = get_stacking_base_datasets(rf_clf, X_train, y_train, X_test, y_test, 7)
dt_train, dt_test = get_stacking_base_datasets(dt_clf, X_train, y_train, X_test, y_test, 7)
ada_train, ada_test = get_stacking_base_datasets(ada_clf, X_train, y_train, X_test, y_test, 7)

KNeighborsClassifier
0
1
2
3
4
5
6
RandomForestClassifier
0
1
2
3
4
5
6
DecisionTreeClassifier
0
1
2
3
4
5
6
AdaBoostClassifier
0
1
2
3
4
5
6


In [17]:
Stacking_fianl_X_train = np.concatenate((knn_train, rf_train, dt_train, ada_train), axis = 1)
Stacking_fianl_X_test = np.concatenate((knn_test, rf_test, dt_test, ada_test), axis = 1)
print(Stacking_fianl_X_train.shape)
print(Stacking_fianl_X_test.shape)

(455, 4)
(114, 4)


In [20]:
lr_final.fit(Stacking_fianl_X_train, y_train)
stack_final = lr_final.predict(Stacking_fianl_X_test)

print(accuracy_score(y_test, stack_final))

0.9649122807017544
